# Heavy-metal concentration prediction with ensemble ML

Trains the four manuscript ML models (RF, XGBoost, LightGBM, SVR), the GP regressor,
and the stacking ensemble; reports R²/RMSE; computes SHAP-based feature importance.

All hyperparameters come from `config/hyperparameters.yaml`.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error
from cmhr.ml_models import build_random_forest, build_xgboost, build_lightgbm, build_svr, build_stacking_ensemble
rng = np.random.default_rng(7); n = 600
X = pd.DataFrame({'NDVI':rng.uniform(-0.2,0.9,n),'elev':rng.uniform(400,1200,n),'pH':rng.uniform(5,8,n),'distance_mining':rng.uniform(0,30,n),'rain':rng.uniform(800,2200,n)})
y = 5 + 8/(X['distance_mining']+1) - 0.5*X['NDVI']*5 + rng.normal(0,1,n)
Xtr,Xte,ytr,yte = train_test_split(X,y,test_size=0.25,random_state=42)
models = {'RF':build_random_forest(),'XGB':build_xgboost(),'LGBM':build_lightgbm(),'SVR':build_svr(),'Stack':build_stacking_ensemble()}
rows=[]
for name, m in models.items():
    if name=='XGB':
        m.fit(Xtr,ytr,eval_set=[(Xte,yte)],verbose=False)
    else:
        m.fit(Xtr,ytr)
    p = m.predict(Xte)
    rows.append({'model':name,'R2':r2_score(yte,p),'RMSE':np.sqrt(mean_squared_error(yte,p))})
pd.DataFrame(rows)

In [ ]:
from cmhr.ml_models import permutation_importance_df
permutation_importance_df(models['RF'], Xte, yte)